In [0]:
%pip install boto3 pandas pyarrow

In [0]:
# Cell 2: Configure AWS Client (带自动去除空格功能)
import boto3

# 1. 重新获取输入值 (添加 .strip() 去除可能存在的空格)
bucket_name = dbutils.widgets.get("s3_bucket").strip()
access_key = dbutils.widgets.get("aws_access_key").strip()
secret_key = dbutils.widgets.get("aws_secret_key").strip()
region = dbutils.widgets.get("aws_region").strip()

# 2. 简单的自我诊断 (不会打印完整密钥，只打印长度)
print(f"检查 Bucket: '{bucket_name}'")
print(f"检查 Access Key 长度: {len(access_key)} (通常是 20 位)")
print(f"检查 Secret Key 长度: {len(secret_key)} (通常是 40 位)")

# 3. 建立连接
if len(access_key) == 20 and len(secret_key) == 40:
    s3_client = boto3.client(
        "s3", aws_access_key_id=access_key, aws_secret_access_key=secret_key, region_name=region
    )
    print("✅ Boto3 客户端已重新初始化 (已去除首尾空格)")
else:
    print("⚠️ 警告：密钥长度看起来不对，请检查是否有误！")

In [0]:
# Cell 3: Fetch Real Data from Coinbase & Upload to S3 (Raw Zone)
import json
import uuid
from datetime import datetime

import requests

# --- 配置参数 ---
CB_BASE = "https://api.exchange.coinbase.com"
PROVIDER = "coinbase"  # 强制只走 Coinbase
PRODUCT_ID = "BTC-USD"  # 交易对
GRANULARITY = 3600  # 3600秒 = 1小时线


def fetch_coinbase_data():
    """
    从 Coinbase Exchange API 获取 BTC-USD 的 K 线数据
    Endpoint: /products/{id}/candles
    """
    if PROVIDER != "coinbase":
        raise ValueError("Configuration Error: Only 'coinbase' provider is supported.")

    url = f"{CB_BASE}/products/{PRODUCT_ID}/candles"
    params = {"granularity": GRANULARITY}
    # Coinbase 有时需要 User-Agent 避免 403
    headers = {"User-Agent": "Databricks-Test-Agent/1.0", "Accept": "application/json"}

    print(f"📡 Requesting from: {url}")

    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
        response.raise_for_status()  # 如果状态码不是 200，直接报错

        # Coinbase 返回的是列表的列表: [time, low, high, open, close, volume]
        raw_list = response.json()

        # 将其转换为更友好的字典格式 (List of Dicts)
        structured_data = []
        for row in raw_list:
            # 为了防止 row 长度不对导致越界，加个简单判断
            if len(row) >= 6:
                structured_data.append(
                    {
                        "provider": PROVIDER,
                        "symbol": PRODUCT_ID,
                        "epoch_time": row[0],
                        "low": row[1],
                        "high": row[2],
                        "open": row[3],
                        "close": row[4],
                        "volume": row[5],
                        "ingest_time": datetime.now().isoformat(),
                    }
                )

        print(f"✅ Successfully fetched {len(structured_data)} candles from Coinbase.")
        return structured_data

    except Exception as e:
        print(f"❌ API Request Failed: {str(e)}")
        raise e  # 强制报错，不进行 fallback


# --- 执行拉取 ---
raw_data = fetch_coinbase_data()

# 生成本次运行的元数据
run_id = str(uuid.uuid4())[:8]
date_str = datetime.now().strftime("%Y-%m-%d")

# --- 上传到 S3 Raw Zone (JSON Lines) ---
# S3 路径: datalake/raw/date=YYYY-MM-DD/run_id_coinbase.json
raw_key = f"datalake/raw/date={date_str}/{run_id}_coinbase.json"

# 转换为 JSON 字符串 (每行一个 JSON 对象)
json_body = "\n".join([json.dumps(record) for record in raw_data])

print(f"Uploading RAW data to: s3://{bucket_name}/{raw_key}")

try:
    # 这里的 s3_client 沿用自 Cell 2
    s3_client.put_object(Body=json_body, Bucket=bucket_name, Key=raw_key)
    print("✅ Raw Upload Success! (真实 Coinbase 数据已上传)")
except Exception as e:
    print(f"❌ Upload Failed: {e}")

In [0]:
# Cell 4: Transform & Load to Curated Zone (Parquet)
import io

import pandas as pd

print("Processing data for Curated Zone...")

# 1. 将 Raw List 转为 Pandas DataFrame
# 注意：这里的 raw_data 是直接引用 Cell 3 内存里的变量
if "raw_data" not in locals() or not raw_data:
    raise ValueError("请先运行 Cell 3 获取 raw_data！")

df = pd.DataFrame(raw_data)

# 2. 数据清洗 / 类型转换
# (A) 添加处理时间
df["ingestion_ts"] = datetime.now()
df["run_id"] = run_id

# (B) 关键步骤：把 Coinbase 的 epoch_time (秒) 转为人类可读的时间
# 这样在做分析时才知道具体是哪根 K 线
df["candle_time"] = pd.to_datetime(df["epoch_time"], unit="s")

# (C) 确保价格字段是浮点数 (虽然通常 API 返回就是 float，但保险起见)
numeric_cols = ["low", "high", "open", "close", "volume"]
for col in numeric_cols:
    df[col] = df[col].astype(float)

# 3. 写入 S3 Curated Zone (Parquet)
# 路径结构: datalake/curated/date=YYYY-MM-DD/run_id_coinbase.parquet
curated_key = f"datalake/curated/date={date_str}/{run_id}_coinbase.parquet"

# 在内存中转为 Parquet 字节流
parquet_buffer = io.BytesIO()
df.to_parquet(parquet_buffer, index=False, engine="pyarrow")

print(f"Uploading CURATED data to: s3://{bucket_name}/{curated_key}")

try:
    s3_client.put_object(Body=parquet_buffer.getvalue(), Bucket=bucket_name, Key=curated_key)
    print("✅ Curated (Parquet) Upload Success! (清洗后的 Parquet 已上传)")
except Exception as e:
    print(f"❌ Upload Failed: {e}")

In [0]:
# Cell 5: Verify Data from S3
print("--- Verifying Data from S3 (Curated Zone) ---")

try:
    # 1. 从 S3 下载 Parquet 文件对象
    response = s3_client.get_object(Bucket=bucket_name, Key=curated_key)

    # 2. 用 Pandas 读取字节流
    read_buffer = io.BytesIO(response["Body"].read())
    df_check = pd.read_parquet(read_buffer)

    print(f"✅ 成功从 S3 读取了 {len(df_check)} 条记录！")
    print("\n--- Top 5 Records (Formatted) ---")

    # 3. 展示前 5 条，重点看时间转换是否成功
    display_cols = ["symbol", "candle_time", "open", "close", "volume"]
    print(df_check[display_cols].head())

    # 如果你是 Notebook 环境，可以直接用 display() 获得美观的表格
    # display(df_check)

except Exception as e:
    print(f"❌ Verification Failed: {e}")

# delete secret key and push to github 

In [0]:
access_key = "REMOVED_FOR_SECURITY"